# 5. Modelagem dos Dados

Após a construção e validação da camada Silver, os dados históricos de produção offshore encontram-se consolidados em uma estrutura única, padronizada e adequada para utilização nas etapas analíticas do projeto.

A próxima etapa consiste na definição do modelo de dados que será utilizado na camada Gold.

A modelagem tem como objetivo organizar os dados de forma adequada às perguntas de negócio, separando os eventos quantitativos de produção das entidades utilizadas para sua análise.

Para isso, será adotada uma abordagem dimensional, com definição de uma tabela fato e das dimensões necessárias para representar os principais eixos de análise do projeto.

Antes da implementação das tabelas Gold, serão avaliados a granularidade dos registros, as entidades disponíveis na camada Silver e os relacionamentos existentes entre elas. Essa análise permitirá definir o modelo com base na estrutura efetivamente observada nos dados.

## 5.1 Objetivo da modelagem dimensional

O objetivo da modelagem é estruturar os dados tratados da camada Silver em um modelo analítico que facilite a realização de agregações, comparações temporais e análises de concentração da produção offshore.

O modelo deverá permitir, principalmente, analisar a produção segundo diferentes perspectivas, como:

- evolução ao longo do tempo;
- distribuição entre estados e bacias sedimentares;
- participação dos diferentes campos;
- concentração da produção ao longo do período analisado.

A definição das tabelas fato e dimensão será realizada a partir dos atributos efetivamente disponíveis na camada Silver. Informações adicionais necessárias para algumas perguntas de negócio, como classificação de pré-sal e identificação de operadores, dependerão da validação de fontes complementares e não serão inferidas a partir de atributos que não representem diretamente essas informações.

## 5.2 Definição da granularidade da tabela fato

A definição da granularidade estabelece o nível de detalhe representado por cada registro da tabela fato e constitui uma das principais decisões da modelagem dimensional.

Durante a avaliação de qualidade, foi observado que a combinação entre `mes_ano`, `poco`, `campo` e `instalacao` não apresentou repetições nas tabelas históricas analisadas.

Como a camada Gold será construída a partir da tabela consolidada `workspace.silver.producao_mar`, essa granularidade será novamente verificada sobre o histórico completo antes da definição da tabela fato.

A validação permitirá confirmar se cada registro da futura tabela fato pode ser interpretado como uma observação mensal de produção associada a uma combinação de poço, campo e instalação.

### 5.2.1 Validação da granularidade na camada Silver

A granularidade será avaliada diretamente na tabela consolidada da camada Silver.

Para isso, os registros serão agrupados pela combinação de `mes_ano`, `poco`, `campo` e `instalacao`. Serão consideradas repetições as combinações que apresentarem mais de um registro.

Além da quantidade de combinações repetidas, será verificado o número máximo de registros encontrado para uma mesma combinação.

Essa validação será realizada antes da definição definitiva do grão da tabela fato.

In [0]:
from pyspark.sql import functions as F

# Carrega a tabela consolidada que servirá como fonte
# para a posterior construção do modelo dimensional.
df_silver = spark.table("workspace.silver.producao_mar")

# Avalia se a combinação observada durante a análise de qualidade
# permanece única após a consolidação de todo o histórico.
df_granularidade = (
    df_silver
    .groupBy(
        "mes_ano",
        "poco",
        "campo",
        "instalacao"
    )
    .count()
)

df_validacao_granularidade = (
    df_granularidade
    .agg(
        F.sum(
            F.when(F.col("count") > 1, 1).otherwise(0)
        ).alias("combinacoes_repetidas"),
        F.max("count").alias("max_registros_por_combinacao")
    )
)

display(df_validacao_granularidade)

combinacoes_repetidas,max_registros_por_combinacao
0,1


#### 5.2.1.1 Resultado da validação da granularidade

A validação realizada sobre a tabela consolidada da camada Silver não identificou combinações repetidas entre `mes_ano`, `poco`, `campo` e `instalacao`.

Foram obtidos os seguintes resultados:

- **0 combinações repetidas**;
- **1 registro como quantidade máxima por combinação**.

O resultado confirma, no histórico consolidado de 2005 a 2025, a granularidade observada anteriormente durante a avaliação de qualidade.

Dessa forma, cada registro representa uma observação mensal associada a uma combinação específica de **poço, campo e instalação**.

### 5.2.2 Definição do grão da tabela fato

Com base na validação realizada, o grão da tabela fato será definido como:

> **uma observação mensal de produção por combinação de poço, campo e instalação.**

Nesse nível de detalhe serão armazenadas as medidas quantitativas presentes na camada Silver, como os volumes de produção de óleo e gás.

A manutenção dessa granularidade preserva o nível de detalhe disponível na fonte e permite que as medidas sejam posteriormente agregadas por diferentes perspectivas analíticas, como tempo, campo, bacia e localização geográfica.

A tabela fato não realizará agregações antecipadas dos volumes, permitindo que as análises da camada Gold sejam construídas a partir do nível mais detalhado disponível no histórico tratado.

A adequação dessa granularidade será complementada por testes adicionais de unicidade antes da definição das entidades dimensionais.

### 5.2.3 Validação de granularidade sem o atributo campo

Após a definição inicial do grão da tabela fato, será realizada uma validação adicional para verificar se o atributo `campo` é necessário para distinguir os registros da camada Silver.

Para isso, será avaliada a combinação entre `mes_ano`, `poco` e `instalacao`, desconsiderando o atributo `campo`.

Serão verificadas a quantidade de combinações que apresentam mais de um registro e a quantidade máxima de registros encontrada para uma mesma combinação.

Essa análise permitirá verificar se a identificação mensal de um poço em uma determinada instalação já é suficiente para representar unicamente os registros ou se o atributo `campo` também é necessário para caracterizar a granularidade observada.

In [0]:
# Avalia se mes_ano, poco e instalacao são suficientes
# para distinguir os registros da camada Silver.

df_granularidade_sem_campo = (
    df_silver
    .groupBy(
        "mes_ano",
        "poco",
        "instalacao"
    )
    .count()
)

df_validacao_sem_campo = (
    df_granularidade_sem_campo
    .agg(
        F.sum(
            F.when(F.col("count") > 1, 1).otherwise(0)
        ).alias("combinacoes_repetidas"),
        F.max("count").alias("max_registros_por_combinacao")
    )
)

display(df_validacao_sem_campo)

combinacoes_repetidas,max_registros_por_combinacao
25639,4


#### 5.2.3.1 Resultado da validação de granularidade sem o atributo campo

A validação da combinação entre `mes_ano`, `poco` e `instalacao`, desconsiderando o atributo `campo`, identificou:

- **25.639 combinações repetidas**;
- até **4 registros para uma mesma combinação**.

O resultado demonstra que a combinação entre mês, poço e instalação não é suficiente para distinguir unicamente os registros da camada Silver.

Em comparação, a inclusão do atributo `campo` na combinação resultou em zero repetições e no máximo um registro por combinação.

Dessa forma, considerando a estrutura observada nos dados, o atributo `campo` é necessário para caracterizar a granularidade dos registros juntamente com `mes_ano`, `poco` e `instalacao`.

### 5.2.4 Investigação das combinações repetidas

A validação anterior demonstrou que a combinação entre `mes_ano`, `poco` e `instalacao` não é suficiente para distinguir unicamente os registros da camada Silver.

Para compreender a origem dessas repetições, serão analisadas as combinações que apresentam mais de um registro, incluindo o atributo `campo`.

Nesta investigação, os registros cuja instalação esteja identificada como `Não Informado` serão desconsiderados temporariamente. Esse filtro será aplicado apenas para fins de diagnóstico, sem exclusão ou alteração dos registros da camada Silver.

O objetivo é verificar se as repetições também ocorrem entre registros que possuem uma instalação identificada e se correspondem a situações em que um mesmo identificador de poço aparece associado a diferentes campos no mesmo mês e instalação.

Essa análise permitirá compreender melhor o papel do atributo `campo` na granularidade observada antes da definição definitiva do grão da tabela fato.

In [0]:
# Identifica todas as combinações repetidas e verifica quantos campos distintos
# estão associados a cada combinação de mês, poço e instalação.
df_combinacoes_repetidas = (
    df_silver
    .filter(F.col("instalacao") != "Não Informado")
    .groupBy(
        "mes_ano",
        "poco",
        "instalacao"
    )
    .agg(
        F.count("*").alias("quantidade_registros"),
        F.countDistinct("campo").alias("quantidade_campos"),
        F.sort_array(
            F.collect_set("campo")
        ).alias("campos_associados")
    )
    .filter(F.col("quantidade_registros") > 1)
)

# Resume o resultado completo da investigação.
resumo_combinacoes = (
    df_combinacoes_repetidas
    .agg(
        F.count("*").alias("combinacoes_repetidas"),
        F.max("quantidade_registros").alias("max_registros_combinacao"),
        F.max("quantidade_campos").alias("max_campos_combinacao")
    )
)

display(resumo_combinacoes)

combinacoes_repetidas,max_registros_combinacao,max_campos_combinacao
25626,4,4


#### 5.2.4.1 Exemplos de combinações associadas a diferentes campos

Para complementar o resumo da investigação, serão apresentados alguns exemplos das combinações que possuem a maior quantidade de campos distintos associados.

A exibição é limitada apenas para facilitar a visualização dos casos, enquanto os indicadores anteriores foram calculados considerando todas as combinações repetidas identificadas.

In [0]:
# Exibe exemplos das combinações com maior quantidade de campos distintos.
display(
    df_combinacoes_repetidas
    .filter(F.col("quantidade_campos") > 1)
    .orderBy(
        F.desc("quantidade_campos"),
        F.desc("quantidade_registros"),
        "mes_ano",
        "poco"
    )
    .limit(10)
)

mes_ano,poco,instalacao,quantidade_registros,quantidade_campos,campos_associados
2015-02-01,3-RJS-159-RJS,PLATAFORMA DE PAMPO 1,4,4,"List(BADEJO, LINGUADO, PAMPO, TRILHA)"
2015-02-01,4-RJS-156-RJS,PETROBRAS 12,4,4,"List(BADEJO, LINGUADO, PAMPO, TRILHA)"
2015-02-01,7-BD-9D-RJS,PETROBRAS 12,4,4,"List(BADEJO, LINGUADO, PAMPO, TRILHA)"
2015-02-01,7-PM-17D-RJS,PLATAFORMA DE PAMPO 1,4,4,"List(BADEJO, LINGUADO, PAMPO, TRILHA)"
2015-03-01,3-LI-4-RJS,PETROBRAS 12,4,4,"List(BADEJO, LINGUADO, PAMPO, TRILHA)"
2015-03-01,3-RJS-159-RJS,PLATAFORMA DE PAMPO 1,4,4,"List(BADEJO, LINGUADO, PAMPO, TRILHA)"
2015-03-01,3-RJS-168-RJS,PETROBRAS 12,4,4,"List(BADEJO, LINGUADO, PAMPO, TRILHA)"
2015-03-01,7-LI-17D-RJS,PETROBRAS 12,4,4,"List(BADEJO, LINGUADO, PAMPO, TRILHA)"
2015-03-01,7-PM-17D-RJS,PLATAFORMA DE PAMPO 1,4,4,"List(BADEJO, LINGUADO, PAMPO, TRILHA)"
2015-04-01,3-LI-4-RJS,PETROBRAS 12,4,4,"List(BADEJO, LINGUADO, PAMPO, TRILHA)"


#### 5.2.4.2 Resultado da investigação das combinações repetidas

A investigação demonstrou que as repetições encontradas ao utilizar apenas `mes_ano`, `poco` e `instalacao` estão associadas à existência de diferentes campos para uma mesma combinação desses atributos.

Considerando o conjunto completo de combinações repetidas analisadas, foram observados casos em que uma mesma combinação de mês, poço e instalação apresenta até quatro registros, correspondentes a quatro campos distintos.

Também foi observado que uma mesma instalação pode estar associada a diferentes campos ao longo do histórico. Parte dessas associações apresenta variações na nomenclatura dos campos, enquanto outras correspondem efetivamente a campos distintos.

Esses resultados reforçam que `campo` não deve ser desconsiderado na definição da granularidade, pois sua remoção faria com que registros distintos fossem tratados como pertencentes à mesma combinação.

Portanto, para os dados analisados, permanece adequada a granularidade definida como uma observação mensal por combinação de **poço, campo e instalação**.

## 5.3 Identificação das entidades dimensionais

Com a granularidade da tabela fato definida, o próximo passo consiste em identificar quais atributos da camada Silver deverão compor as dimensões do modelo analítico.

A presença de um atributo descritivo na fonte não implica necessariamente a criação de uma dimensão independente. A definição das dimensões deve considerar tanto sua relevância para as perguntas de negócio quanto as relações existentes entre as entidades ao longo do histórico.

A camada Silver contém atributos relacionados a diferentes perspectivas de análise, incluindo tempo, localização geográfica e estrutura produtiva, como `estado`, `bacia`, `campo`, `poco` e `instalacao`.

Antes de definir a estrutura dimensional, serão analisadas as relações entre esses atributos para verificar suas cardinalidades e identificar possíveis associações múltiplas.

Essa avaliação servirá de base para decidir quais entidades serão representadas como dimensões independentes e quais atributos poderão ser agrupados em uma mesma estrutura dimensional.

### 5.3.1 Identificação dos candidatos a dimensões

A partir da granularidade definida para a tabela fato e das perguntas de negócio do projeto, foram identificados os principais atributos descritivos que podem representar perspectivas de análise no modelo dimensional.

Na camada Silver, os atributos relevantes podem ser agrupados inicialmente nas seguintes perspectivas:

- **Tempo:** representado por `mes_ano` e `ano`;
- **Localização e estrutura produtiva:** representadas por `estado`, `bacia` e `campo`;
- **Poço:** representado por `poco`;
- **Instalação:** representada por `instalacao`.

Os atributos quantitativos de produção, por sua vez, representam medidas associadas aos eventos registrados e, portanto, são candidatos a permanecer na tabela fato.

A identificação dessas entidades constitui uma proposta inicial. Antes da definição definitiva das dimensões, serão consideradas as relações observadas entre os atributos e sua utilidade para responder às perguntas de negócio.

### 5.3.2 Avaliação das relações entre campo, bacia e estado

Os atributos `estado`, `bacia` e `campo` representam diferentes níveis de caracterização geográfica e produtiva dos registros.

Antes de definir se esses atributos deverão compor dimensões independentes ou uma estrutura dimensional conjunta, serão avaliadas suas relações no histórico consolidado.

Serão verificadas:

- a quantidade de bacias distintas associadas a cada campo;
- a quantidade de estados distintos associada a cada campo.

O objetivo é identificar se um campo mantém associações únicas com bacia e estado ao longo do histórico ou se existem campos relacionados a múltiplos valores desses atributos.

In [0]:
# Avalia se cada campo está associado a uma única bacia
# e a um único estado ao longo do histórico.

df_relacoes_campo = (
    df_silver
    .filter(F.col("campo").isNotNull())
    .groupBy("campo")
    .agg(
        F.countDistinct("bacia").alias("quantidade_bacias"),
        F.countDistinct("estado").alias("quantidade_estados")
    )
)

df_resumo_relacoes_campo = (
    df_relacoes_campo
    .agg(
        F.count("*").alias("total_campos"),
        F.sum(
            F.when(F.col("quantidade_bacias") > 1, 1).otherwise(0)
        ).alias("campos_multiplas_bacias"),
        F.sum(
            F.when(F.col("quantidade_estados") > 1, 1).otherwise(0)
        ).alias("campos_multiplos_estados"),
        F.max("quantidade_bacias").alias("max_bacias_por_campo"),
        F.max("quantidade_estados").alias("max_estados_por_campo")
    )
)

display(df_resumo_relacoes_campo)

total_campos,campos_multiplas_bacias,campos_multiplos_estados,max_bacias_por_campo,max_estados_por_campo
144,0,3,1,2


#### 5.3.2.1 Resultado da avaliação das relações

Após a correção das inconsistências de codificação identificadas nos atributos categóricos da camada Silver, a avaliação das relações entre campo, bacia e estado foi executada novamente.

Foram identificados **144 campos distintos** no histórico consolidado.

Nenhum campo apresentou associação com mais de uma bacia, sendo observada no máximo uma bacia distinta para cada campo. Esse resultado indica uma relação consistente entre `campo` e `bacia` nos dados tratados.

Em relação ao atributo `estado`, apenas **3 campos** apresentaram associação com mais de um valor, com no máximo dois estados distintos por campo.

A redução das associações múltiplas em relação à avaliação anterior confirma que parte das inconsistências observadas estava relacionada à codificação incorreta de caracteres nos valores categóricos.

Os três casos remanescentes de associação entre um campo e múltiplos estados serão examinados antes da definição da estrutura dimensional.

### 5.3.3 Investigação dos campos associados a múltiplos estados

Após a correção dos valores categóricos, apenas três campos permaneceram associados a mais de um estado no histórico consolidado.

Como cada campo passou a apresentar associação com uma única bacia, a investigação será direcionada somente aos casos de múltiplos estados.

O objetivo é identificar os campos envolvidos e os respectivos valores de estado, verificando se as associações correspondem a diferentes estados efetivamente informados ou à presença de valores sem identificação.

In [0]:
# Identifica somente os campos associados a mais de um estado
# após a correção dos valores categóricos da camada Silver.

df_campos_multiplos_estados = (
    df_relacoes_campo
    .filter(F.col("quantidade_estados") > 1)
    .select("campo")
    .join(
        df_silver
        .select(
            "campo",
            "bacia",
            "estado"
        )
        .distinct(),
        on="campo",
        how="inner"
    )
    .orderBy(
        "campo",
        "estado"
    )
)

display(df_campos_multiplos_estados)

campo,bacia,estado
NORDESTE DE SAPINHOÁ,Santos,Não Informado
NORDESTE DE SAPINHOÁ,Santos,São Paulo
NOROESTE DE SAPINHOÁ,Santos,Não Informado
NOROESTE DE SAPINHOÁ,Santos,São Paulo
SUDOESTE DE SAPINHOÁ,Santos,Não Informado
SUDOESTE DE SAPINHOÁ,Santos,São Paulo


#### 5.3.3.1 Resultado da investigação dos campos associados a múltiplos estados

A investigação dos três campos que permaneceram associados a mais de um valor de estado demonstrou que as ocorrências não correspondem à associação de um mesmo campo a dois estados distintos efetivamente identificados.

Nos casos observados, os campos apresentam registros associados ao estado de `São Paulo` e registros cujo estado está classificado como `Não Informado`.

Dessa forma, a multiplicidade observada está relacionada à ausência de informação em parte dos registros, e não à associação dos campos a diferentes estados identificados.

Optou-se por preservar os valores `Não Informado` conforme disponibilizados na fonte, sem realizar imputação do estado, evitando introduzir informações inferidas na camada Silver.

Considerando também que nenhum campo apresentou associação com mais de uma bacia após a correção dos valores categóricos, os resultados sustentam uma relação consistente entre campo, bacia e estado para fins de modelagem, ressalvados os registros em que o estado não foi informado.

## 5.4 Definição do modelo dimensional

Com a granularidade da tabela fato validada e as relações entre os principais atributos descritivos analisadas, é possível definir a estrutura dimensional que orientará a construção da camada Gold.

O modelo será organizado segundo uma abordagem dimensional, separando as medidas quantitativas de produção das principais perspectivas utilizadas para análise.

A tabela fato manterá o nível de detalhe mensal por combinação de poço, campo e instalação, enquanto as dimensões representarão os elementos descritivos necessários para contextualizar esses registros.

A definição busca preservar o nível de detalhe disponível na camada Silver e, ao mesmo tempo, facilitar agregações e análises relacionadas às perguntas de negócio do projeto.

### 5.4.1 Definição da tabela fato

A tabela `fato_producao` será responsável por armazenar as medidas quantitativas necessárias para responder às perguntas de negócio definidas no projeto.

Seu grão será:

> **uma observação mensal de produção por combinação de poço, campo e instalação.**

Embora a camada Silver preserve os 13 atributos quantitativos existentes na fonte, a camada Gold será orientada às necessidades analíticas do projeto. Dessa forma, serão mantidas na tabela fato apenas duas medidas:

- `producao_oleo_mmbbl`: produção de óleo expressa em milhões de barris (MMbbl), obtida a partir da conversão da produção originalmente disponibilizada em metros cúbicos;
- `producao_gas_bilhoes_m3`: produção total de gás natural expressa em bilhões de metros cúbicos, calculada a partir da soma da produção de gás associado e gás não associado e da conversão da unidade disponibilizada na fonte.

As conversões de unidade serão realizadas exclusivamente na camada Gold, preservando na camada Silver os valores e unidades padronizados a partir da estrutura disponibilizada pela fonte.

Neste trabalho, a produção de petróleo será representada exclusivamente pela produção de óleo, sem inclusão do condensado.

A produção total de gás natural representará o volume bruto produzido. Por esse motivo, as medidas relacionadas à injeção de gás não serão utilizadas para reduzir o volume produzido, uma vez que o objetivo do projeto é analisar produção e não disponibilidade ou oferta de gás ao mercado.

As demais medidas quantitativas permanecerão preservadas na camada Silver, mas não serão incorporadas à tabela fato da camada Gold por não serem necessárias às análises propostas.

Além das duas medidas, a tabela fato conterá as chaves utilizadas para estabelecer os relacionamentos com as dimensões de tempo, campo, poço e instalação.

### 5.4.2 Definição das dimensões

Considerando as entidades disponíveis na camada Silver, as relações identificadas durante a análise e as necessidades das perguntas de negócio, são propostas quatro dimensões principais:

**Dimensão Tempo (`dim_tempo`)**

Representará a perspectiva temporal dos registros e será derivada do atributo `mes_ano`. Permitirá realizar análises mensais e anuais da produção ao longo do período de 2005 a 2025.

**Dimensão Campo (`dim_campo`)**

Representará os campos de produção e incorporará os atributos de bacia e estado. A análise realizada demonstrou que cada campo está associado a uma única bacia nos dados tratados e que as associações múltiplas com estados remanescentes estão relacionadas à presença do valor `Não Informado`.

A incorporação de `bacia` e `estado` na dimensão de campo permite analisar a produção nessas diferentes perspectivas sem a necessidade de criar dimensões adicionais para cada atributo.

**Dimensão Poço (`dim_poco`)**

Representará os poços presentes nos registros de produção, preservando essa entidade separadamente no modelo em razão de sua participação na granularidade da tabela fato.

**Dimensão Instalação (`dim_instalacao`)**

Representará as instalações associadas aos registros de produção. A dimensão será mantida separadamente porque uma mesma instalação pode estar relacionada a diferentes campos ao longo do histórico, conforme observado durante a avaliação da granularidade.

### 5.4.3 Regra para consolidação dos atributos da dimensão Campo

Durante a análise das relações entre campo, bacia e estado, foram identificados três campos que possuem registros associados ao estado de `São Paulo` e registros cujo estado está classificado como `Não Informado`.

Para garantir uma única linha por campo na dimensão, será adotada uma regra de consolidação exclusivamente na camada Gold.

Quando existir um estado efetivamente informado para determinado campo, esse valor será utilizado como atributo da dimensão. O valor `Não Informado` será mantido somente para campos que não apresentem nenhum estado identificado no histórico.

Essa regra não modifica os registros da camada Silver nem substitui os valores presentes na fonte. Seu objetivo é apenas estabelecer uma representação dimensional única para cada campo na camada analítica.

### 5.4.4 Validação dos atributos das dimensões Poço e Instalação

Antes da definição final das dimensões `dim_poco` e `dim_instalacao`, serão avaliados os valores disponíveis nos atributos `poco` e `instalacao` da camada Silver.

A validação tem como objetivo identificar a quantidade de valores distintos, a presença de valores nulos e a ocorrência da categoria `Não Informado`.

Essa verificação permitirá definir como esses casos deverão ser representados nas respectivas dimensões da camada Gold, sem modificar os registros preservados na camada Silver.

In [0]:
# Avalia os atributos que serão utilizados na construção
# das dimensões de poço e instalação.

resultado_dimensoes = (
    df_silver
    .agg(
        F.countDistinct("poco").alias("pocos_distintos"),
        F.sum(F.col("poco").isNull().cast("int")).alias("pocos_nulos"),
        F.sum(
            (F.col("poco") == "Não Informado").cast("int")
        ).alias("registros_poco_nao_informado"),

        F.countDistinct("instalacao").alias("instalacoes_distintas"),
        F.sum(F.col("instalacao").isNull().cast("int")).alias("instalacoes_nulas"),
        F.sum(
            (F.col("instalacao") == "Não Informado").cast("int")
        ).alias("registros_instalacao_nao_informado")
    )
)

display(resultado_dimensoes)

pocos_distintos,pocos_nulos,registros_poco_nao_informado,instalacoes_distintas,instalacoes_nulas,registros_instalacao_nao_informado
2503,0,0,231,4095,98289


#### 5.4.4.1 Resultado da validação dos atributos das dimensões Poço e Instalação

A avaliação identificou **2.503 poços distintos** no histórico consolidado. O atributo `poco` não apresentou valores nulos nem registros classificados como `Não Informado`, permitindo sua utilização direta na construção da dimensão de poços.

Para o atributo `instalacao`, foram identificados **231 valores distintos**, além de **4.095 registros nulos** e **98.289 registros classificados como `Não Informado`**.

A presença desses registros exige um tratamento específico na construção do modelo dimensional para evitar referências sem correspondência na tabela fato.

Dessa forma, na camada Gold, os registros cuja instalação seja nula serão associados à categoria `Não Informado`, já existente nos dados históricos. Essa regra será aplicada exclusivamente para fins de relacionamento dimensional, sem modificar os valores preservados na camada Silver.

Com essa definição, as dimensões de poço e instalação podem ser construídas mantendo uma referência válida para todos os registros da tabela fato.

### 5.4.5 Estrutura final do modelo dimensional

Com base na granularidade validada, nas relações identificadas entre os atributos da camada Silver e nas perguntas de negócio do projeto, o modelo dimensional será composto por uma tabela fato e quatro dimensões.

A estrutura proposta é:

**`fato_producao`**

Tabela central do modelo, com granularidade mensal por combinação de poço, campo e instalação.

Contém:
- `sk_tempo`: chave estrangeira para `dim_tempo`;
- `sk_campo`: chave estrangeira para `dim_campo`;
- `sk_poco`: chave estrangeira para `dim_poco`;
- `sk_instalacao`: chave estrangeira para `dim_instalacao`;
- `producao_oleo_mmbbl`: produção de óleo expressa em milhões de barris (MMbbl), obtida a partir da conversão da produção originalmente disponibilizada em metros cúbicos;
- `producao_gas_bilhoes_m3`: produção total de gás natural expressa em bilhões de metros cúbicos, calculada a partir da soma da produção de gás associado e gás não associado e da conversão da unidade disponibilizada na fonte.

**`dim_tempo`**

Contém:
- `sk_tempo`: chave substituta e chave primária;
- `mes_ano`: data de referência mensal;
- `ano`: ano de referência;
- `mes`: número do mês;
- `nome_mes`: nome do mês.

**`dim_campo`**

Contém:
- `sk_campo`: chave substituta e chave primária;
- `campo`;
- `bacia`;
- `estado`.

**`dim_poco`**

Contém:
- `sk_poco`: chave substituta e chave primária;
- `poco`.

**`dim_instalacao`**

Contém:
- `sk_instalacao`: chave substituta e chave primária;
- `instalacao`.

Os registros cuja instalação esteja nula na camada Silver serão associados à categoria `Não Informado` durante a construção da camada Gold.

As dimensões utilizarão chaves substitutas geradas na camada Gold. Essas chaves atuarão como chaves primárias nas dimensões e como chaves estrangeiras na tabela fato, estabelecendo os relacionamentos do modelo estrela.

### 5.4.6 Relacionamentos do modelo

O modelo será estruturado em formato estrela, tendo `fato_producao` como tabela central.

Cada registro da tabela fato estará relacionado a uma ocorrência das dimensões de tempo, campo, poço e instalação por meio das respectivas chaves substitutas.

Os relacionamentos previstos são:

- `dim_tempo` **1:N** `fato_producao`;
- `dim_campo` **1:N** `fato_producao`;
- `dim_poco` **1:N** `fato_producao`;
- `dim_instalacao` **1:N** `fato_producao`.

Nesse desenho, cada registro dimensional pode estar associado a diversos registros históricos da tabela fato, enquanto cada registro da fato referencia uma única ocorrência de cada dimensão.

Essa estrutura permite analisar as medidas de produção ao longo do tempo e segundo diferentes perspectivas, como campo, bacia, estado, poço e instalação.

## 5.5 Representação do modelo dimensional

A estrutura definida pode ser representada por um modelo estrela, no qual a tabela `fato_producao` ocupa a posição central e se relaciona diretamente com as quatro dimensões: tempo, campo, poço e instalação.

A Figura 1 apresenta a representação do modelo dimensional proposto para a camada Gold, incluindo as tabelas, seus principais atributos, as chaves primárias e estrangeiras e os relacionamentos estabelecidos.

**Figura 1 — Modelo dimensional em estrela da camada Gold.**
*![Modelo dimensional](https://raw.githubusercontent.com/renatanegreiross/Sprint2_EngenhariadeDados_MVP/main/doc/modelo_dimensional.png)**  
*Fonte: elaboração própria.*

### 5.5.1 Resultado da representação do modelo

O modelo dimensional definido apresenta estrutura em estrela, com a tabela `fato_producao` posicionada no centro e relacionada às dimensões `dim_tempo`, `dim_campo`, `dim_poco` e `dim_instalacao`.

Cada dimensão possui uma chave substituta utilizada como chave primária, enquanto a tabela fato armazena essas mesmas chaves como chaves estrangeiras, estabelecendo relacionamentos do tipo **1:N** entre cada dimensão e a fato.

A tabela fato foi mantida enxuta e orientada às perguntas de negócio, contendo somente as medidas de produção de óleo e produção total de gás natural necessárias às análises propostas.

A separação entre dimensões e fato reduz a repetição dos atributos descritivos presentes na camada Silver e organiza os dados de forma adequada às consultas analíticas que serão realizadas na camada Gold.

## 5.6 Conclusão da modelagem dos dados

A análise da granularidade e dos relacionamentos existentes na camada Silver permitiu definir um modelo dimensional em estrela para representar os dados de produção offshore.

A granularidade da tabela fato foi estabelecida como uma observação mensal por combinação de poço, campo e instalação. Os testes realizados mostraram que a retirada do campo dessa composição gera repetições, confirmando sua relevância para a identificação da granularidade observada nos dados.

O modelo foi estruturado em quatro dimensões — tempo, campo, poço e instalação — e uma tabela fato de produção. Estado e bacia foram incorporados como atributos da dimensão campo, considerando as relações observadas durante as validações. Os casos de ausência de informação identificados na Silver serão tratados na construção dimensional da Gold sem alterar os dados preservados na camada anterior.

A tabela fato armazenará as medidas `producao_oleo_mmbbl` e `producao_gas_bilhoes_m3`, sendo esta última derivada da produção de gás associado e não associado. As demais medidas disponíveis na camada Silver permanecerão preservadas, mas não serão incorporadas à Gold por não serem necessárias para responder às perguntas de negócio definidas neste projeto.

Com a definição do modelo dimensional concluída, a próxima etapa consiste na implementação das dimensões e da tabela fato na camada Gold, incluindo a geração das chaves substitutas e a validação dos relacionamentos entre as tabelas.